# Quantization-aware training: khi PTQ không đủ

Ở 2-3 bit, PTQ (kể cả GPTQ) làm GPT mini mất nhiều bpc. **Quantization-aware training** (QAT) cho model "thấy" phép làm tròn
ngay trong lúc train: forward dùng trọng số đã fake-quantize, backward cho gradient đi thẳng qua phép làm tròn (STE, bài
Basic 03) để cập nhật bản trọng số float32 gốc. Bạn viết các lớp QAT, fine-tune GPT mini của module 05 vài trăm bước, và
tách bạch phần cải thiện do QAT với phần do "được train thêm" bằng một thí nghiệm đối chứng.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm/lớp. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` nạp checkpoint, train, đánh giá; test bỏ qua chúng.
- Checkpoint: như bài Intermediate 01. Có GPU thì chạy bằng kernel `ai-gpu` (nhanh hơn nhiều); CPU tự giảm số bước.
- Chấm: `uv run pytest tracks/ai-engineer/modules/10-quantization/exercises/intermediate/03-qat -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, kiểm Wikipedia tiếng Việt (tải một lần bằng env gốc), chỗ để checkpoint
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
WIKI = ROOT / "data/raw/wikipedia-vi"
if not (WIKI.is_dir() and any(WIKI.iterdir())):
    from data.download_dataset import ensure

    ensure("wikipedia-vi")
SHARD = WIKI / "20231101.vi/train-00002-of-00004.parquet"
CKPT = Path(os.environ.get("GPT_MINI_CKPT", ROOT / "data/raw/gpt-mini-vi/gpt_mini_vi.pt"))
FALLBACK = Path(os.environ.get("GPT_MINI_FALLBACK", ROOT / "data/raw/gpt-mini-vi/fallback_small.pt"))
print("checkpoint:", CKPT, "có" if CKPT.exists() else "chưa có")

In [ ]:
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. Fake quantization có STE theo nhóm

`fake_quant_ste(w, bits, group_size)`: forward trả `w` đã quantize rồi dequantize **đối xứng** (absmax) theo nhóm `group_size`
cột liền nhau của một hàng (như bài Basic 02, khối toàn 0 dùng scale 1); scale tính từ `w` nhưng **không** nhận gradient;
backward là STE đồng nhất: gradient theo `w` bằng đúng gradient theo đầu ra. Không chia hết thì `ValueError`.

In [ ]:
def fake_quant_ste(w: torch.Tensor, bits: int, group_size: int) -> torch.Tensor:
    out_f, in_f = w.shape
    if in_f % group_size:
        raise ValueError(f"in_features {in_f} không chia hết cho nhóm {group_size}")
    qmax = 2 ** (bits - 1) - 1
    g = w.reshape(out_f, in_f // group_size, group_size)
    scale = g.detach().abs().amax(dim=-1, keepdim=True) / qmax
    scale = torch.where(scale == 0, torch.ones_like(scale), scale)
    q = (torch.clamp(torch.round(g.detach() / scale), -qmax, qmax) * scale).reshape(out_f, in_f)
    return w + (q - w).detach()

## 2. Chuẩn bị và chuyển đổi model

- `QATLinear(linear, bits, group_size)`: một `nn.Module` giữ **chính** `nn.Parameter` `weight` và `bias` của `linear` (không
  sao chép), có thuộc tính `bits`, `group_size`, `in_features`, `out_features`; `forward(x)` là `F.linear(x, fake_quant_ste(weight), bias)`.
- `prepare_qat(model, bits, group_size)`: bản sao của model (gốc không đổi) trong đó mọi `nn.Linear` được thay bằng `QATLinear`
  bọc nó, **trừ** `nn.Linear` dùng chung trọng số với một `nn.Embedding` (`lm_head` của GPT mini giữ nguyên float32, như
  mặc định của bài Intermediate 01). Số tham số không đổi, weight tying giữ nguyên.
- `convert_qat(model)`: bản sao trong đó mỗi `QATLinear` thành lại một `nn.Linear` thường có trọng số là kết quả fake-quantize
  (đã đóng băng, không còn STE) và cùng bias. Đây là model bạn sẽ xuất ra định dạng int thật.

In [ ]:
class QATLinear(nn.Module):
    def __init__(self, linear: nn.Linear, bits: int, group_size: int):
        super().__init__()
        self.weight = linear.weight
        self.bias = linear.bias
        self.bits, self.group_size = bits, group_size
        self.in_features, self.out_features = linear.in_features, linear.out_features

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.linear(x, fake_quant_ste(self.weight, self.bits, self.group_size), self.bias)


def _swap(model: nn.Module, make) -> None:
    embedding_weights = {id(m.weight) for m in model.modules() if isinstance(m, nn.Embedding)}
    for parent in list(model.modules()):
        for name, child in list(parent.named_children()):
            new = make(child, embedding_weights)
            if new is not None:
                setattr(parent, name, new)


def prepare_qat(model: nn.Module, bits: int, group_size: int) -> nn.Module:
    q = copy.deepcopy(model)
    _swap(q, lambda c, emb: QATLinear(c, bits, group_size)
          if type(c) is nn.Linear and id(c.weight) not in emb else None)
    return q


def convert_qat(model: nn.Module) -> nn.Module:
    q = copy.deepcopy(model)

    def make(c, emb):
        if not isinstance(c, QATLinear):
            return None
        lin = nn.Linear(c.in_features, c.out_features, bias=c.bias is not None)
        with torch.no_grad():
            lin.weight.copy_(fake_quant_ste(c.weight, c.bits, c.group_size))
            if c.bias is not None:
                lin.bias.copy_(c.bias)
        return lin.to(c.weight.device)

    _swap(q, make)
    return q

## 3. Thí nghiệm

Không chấm. Ba model được so trên cùng validation, cùng cấu hình lượng tử:

1. **PTQ**: quantize checkpoint gốc.
2. **QAT**: `prepare_qat`, fine-tune, `convert_qat`.
3. **Đối chứng**: fine-tune float32 đúng như (2) (cùng dữ liệu, số bước, learning rate, seed) rồi mới PTQ. Không có dòng này thì
   không biết QAT thắng nhờ "thấy" lượng tử hay chỉ nhờ được train thêm.

Thời gian: GPU vài phút; CPU với checkpoint cấu hình PC khoảng 15 phút (notebook tự giảm số bước).

In [ ]:
# explore: nạp lớp GPT (Basic 01) và các hàm dữ liệu, đánh giá (Intermediate 01) của module 05: bài của bạn nếu đã làm, không thì lời giải
import importlib.util

from testkit import Exercise

M05 = ROOT / "tracks/ai-engineer/modules/05-transformers-llm-architecture"


def first_working(loaders, probe):
    for load in loaders:
        module = load()
        try:
            probe(module)
            return module
        except NotImplementedError:
            continue
    raise RuntimeError("không nạp được code module 05")


def py_loader(path):
    def load():
        spec = importlib.util.spec_from_file_location("gpt_mini", path)
        module = importlib.util.module_from_spec(spec)
        sys.modules["gpt_mini"] = module
        spec.loader.exec_module(module)
        return module
    return load


def nb_loader(target):
    def load():
        ex05 = Exercise(str(M05 / "exercises/intermediate/01-train-gpt-wikipedia/tests/test_train_gpt.py"))
        ex05.target = target
        return ex05.notebook("train_gpt.ipynb")
    return load


gpt_mini = first_working(
    [py_loader(M05 / "exercises/basic/01-gpt-mini/starter/gpt.py"), py_loader(M05 / "solutions/basic/01-gpt-mini/gpt.py")],
    lambda m: m.GPT(m.GPTConfig(vocab_size=10, block_size=4, n_layer=1, n_head=1, n_embd=8))(torch.zeros(1, 2, dtype=torch.long)),
)
m05 = first_working(
    [nb_loader(M05 / "exercises/intermediate/01-train-gpt-wikipedia/starter"),
     nb_loader(M05 / "solutions/intermediate/01-train-gpt-wikipedia")],
    lambda m: (m.sliding_windows(10, 4, 2), m.split_articles(["a", "b", "c"], 1, 1, 0), m.nll_to_metrics(1.0, 1, 1),
               m.lr_at(0, 1e-3, 10, 100, 1e-4)),
)
print("GPT:", gpt_mini.__file__)
print("hàm đánh giá:", m05.__file__)

In [ ]:
# explore: nạp checkpoint gpt-mini-v1 của bạn; chưa có thì train một GPT nhỏ dự phòng (CPU khoảng 4 phút, lưu lại để dùng tiếp)
import dataclasses
import time

from tokenizers import Tokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(4)
articles = m05.long_articles(SHARD, 2000)
train_articles, val_articles = m05.split_articles(articles, 1000, 100, seed=0)


def load_gpt_mini(path):
    raw = torch.load(path, map_location="cpu", weights_only=True)
    if raw.get("format") != "gpt-mini-v1":
        raise ValueError(f"{path} không phải định dạng gpt-mini-v1")
    model = gpt_mini.GPT(gpt_mini.GPTConfig(**raw["config"]))
    model.load_state_dict(raw["state_dict"])
    return model.eval(), Tokenizer.from_str(raw["tokenizer"])


def train_fallback(path, steps=600):
    tok = m05.train_bpe("\n\n".join(train_articles), 8000)
    ids = torch.tensor(tok.encode("\n\n".join(train_articles)).ids, dtype=torch.long)
    torch.manual_seed(0)
    config = gpt_mini.GPTConfig(vocab_size=8000, block_size=128, n_layer=4, n_head=4, n_embd=128)
    model = gpt_mini.GPT(config)
    opt = m05.configure_optimizer(model, 1e-3, 0.1)
    g = torch.Generator().manual_seed(0)
    for step in range(steps):
        for group in opt.param_groups:
            group["lr"] = m05.lr_at(step, 1e-3, 100, steps, 1e-4)
        x, y = m05.get_batch(ids, 32, 128, g)
        _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"format": "gpt-mini-v1", "config": dataclasses.asdict(config),
                "state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
                "tokenizer": tok.to_str(), "step": steps, "metrics": {}}, path)


source = CKPT if CKPT.exists() else FALLBACK
if not source.exists():
    print("chưa có checkpoint module 05: train GPT nhỏ dự phòng...")
    start = time.perf_counter()
    train_fallback(FALLBACK)
    print(f"xong sau {time.perf_counter() - start:.0f} s, lưu ở {FALLBACK}")
model, tok = load_gpt_mini(source)
BLOCK = model.config.block_size
print(source.name, model.config, f"{model.num_params():,} tham số")

In [ ]:
# explore: validation của module 04/05; CPU chỉ lấy 20 bài đầu cho nhanh, GPU lấy đủ 100 bài
N_VAL = 100 if DEVICE == "cuda" else 20
val_text = "\n\n".join(val_articles[:N_VAL])
val_ids = torch.tensor(tok.encode(val_text).ids, dtype=torch.long)


def val_bpc(m) -> float:
    total, n = m05.evaluate_nll(m.to(DEVICE), val_ids, BLOCK, BLOCK // 2, device=DEVICE)
    return m05.nll_to_metrics(total, n, len(val_text))["bpc"]


start = time.perf_counter()
BASE_BPC = val_bpc(model)
print(f"{N_VAL} bài, {len(val_text):,} ký tự, {len(val_ids):,} token | float32 bpc {BASE_BPC:.4f}"
      f" ({time.perf_counter() - start:.1f} s mỗi lần đánh giá trên {DEVICE})")

In [ ]:
# explore: dữ liệu fine-tune (tập train module 04/05) và vòng train dùng chung cho QAT và đối chứng
BITS, GROUP = 2, 32
STEPS = 300 if DEVICE == "cuda" else 120
BATCH, LR = 16, 3e-4
train_ids = torch.tensor(tok.encode("\n\n".join(train_articles)).ids, dtype=torch.long)


def finetune(m, steps=STEPS, seed=0):
    m = m.to(DEVICE).train()
    opt = torch.optim.AdamW(m.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=0.0)
    g = torch.Generator().manual_seed(seed)
    losses = []
    for step in range(steps):
        for group in opt.param_groups:
            group["lr"] = m05.lr_at(step, LR, 20, steps, LR / 10)
        x, y = m05.get_batch(train_ids, BATCH, BLOCK, g)
        _, loss = m(x.to(DEVICE), y.to(DEVICE))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt.step()
        losses.append(loss.item())
    return m.eval(), losses


def ptq(m):
    return convert_qat(prepare_qat(m, BITS, GROUP))


print(f"{len(train_ids):,} token train | int{BITS} nhóm {GROUP} | {STEPS} bước batch {BATCH} trên {DEVICE}")
results = {"float32 gốc": BASE_BPC, "PTQ": val_bpc(ptq(model))}
print(results)

In [ ]:
# explore: QAT và đối chứng float32 + PTQ
start = time.perf_counter()
torch.manual_seed(0)
qat_model, qat_losses = finetune(prepare_qat(model, BITS, GROUP))
results["QAT"] = val_bpc(convert_qat(qat_model))
t_qat = time.perf_counter() - start
start = time.perf_counter()
torch.manual_seed(0)
ft_model, ft_losses = finetune(copy.deepcopy(model))
results["fine-tune float32"] = val_bpc(ft_model)
results["fine-tune float32 rồi PTQ"] = val_bpc(ptq(ft_model))
print(f"QAT {t_qat:.0f} s, đối chứng {time.perf_counter() - start:.0f} s")
for k, v in results.items():
    print(f"{k:28s} bpc {v:.4f}  (Δ so với gốc {v - BASE_BPC:+.4f})")

In [ ]:
# explore: loss train của hai lần fine-tune
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 3.2))
plt.plot(qat_losses, lw=0.7, label=f"QAT int{BITS} (loss của model đã fake-quantize)")
plt.plot(ft_losses, lw=0.7, label="fine-tune float32")
plt.xlabel("bước"); plt.ylabel("loss train"); plt.legend(); plt.tight_layout(); plt.show()

## 4. Đọc kết quả

So với mục 6 phần lý thuyết.

1. Xếp năm dòng theo bpc. QAT lấy lại được bao nhiêu phần khoảng cách giữa PTQ và float32?
2. So QAT với "fine-tune float32 rồi PTQ": nếu hai số gần nhau, cải thiện đến từ việc được train thêm; nếu QAT tốt hơn rõ, model
   đã học cách đặt trọng số hợp lưới lượng tử.
3. Đường loss: loss QAT bắt đầu cao hơn float32 bao nhiêu, và khoảng cách đó thay đổi thế nào qua các bước.
4. Cái giá của QAT: thời gian train, bộ nhớ (vẫn giữ trọng số float32 và trạng thái optimizer), và cần dữ liệu train. Khi nào
   đáng trả cái giá đó, khi nào GPTQ là đủ.

Sau khi chạy hết notebook, làm phần **trắc nghiệm** của bài trong app.